In [ ]:
import xarray as xr
import numpy as np
import os
import geopandas as gpd
import pandas as pd
from multiprocessing import Pool
import netCDF4

# This script checks if there are default values in the dataset that needs to be converted to NA according to Edition4A documentation

# The files were in netCDF3zip format, after manually unzipping, they are netCDF3 now
def clean_nc_file(file):
   with xr.open_dataset(file) as ds:
        sw_flux = ds["CERES_SW_TOA_flux___upwards"].values
        if (sw_flux>=32767).sum() != 0: 
            print((sw_flux>=32767).sum())
        
        net_sw_flux = ds["CERES_net_SW_surface_flux___Model_B"].values
        if (net_sw_flux>=32767).sum() != 0: 
            print((net_sw_flux>=32767).sum())

def inspect_variable(file):
    # with xr.open_dataset(file, mask_and_scale=False) as ds:
    #     for var_name in ["CERES_SW_TOA_flux___upwards","CERES_net_SW_surface_flux___Model_B"]:

    #         values = ds[var_name].values
    #         print(f"total size: {values.size}")
    #         print(f"NaN count: {np.isnan(values).sum()}")
    #         print(f"non-NaN count: {(~np.isnan(values)).sum()}")

    #         if (~np.isnan(values)).any():
    #             print(f"min (ignoring NaN): {np.nanmin(values)}")
    #             print(f"max (ignoring NaN): {np.nanmax(values)}")

    ds = netCDF4.Dataset(file)
    for var_name in ["CERES_SW_TOA_flux___upwards","CERES_net_SW_surface_flux___Model_B"]:
        var = ds.variables[var_name]
        var.set_auto_maskandscale(False)
        raw = var[:]
        if raw.dtype != "float32":
            print(f"dtype: {raw.dtype}")
        if np.nanmax(raw)> 1400:
            print(f"min: {np.nanmin(raw)}, max: {np.nanmax(raw)}")
      
        
if __name__ == "__main__":

    # load all files
    os.chdir("/Users/anora/Team MG Dropbox/Wanru Wu/Cloudseeding_Anora/SSF/raw/terra")

    path = os.getcwd() 
    
    # Get the list of all files and directories 
    all_files = os.listdir(path) 
    data_files_terra = [file for file in all_files if file.find("CERES_SSF_Terra-XTRK_Edition4A_Subset")!=-1]

    for file in data_files_terra:
        inspect_variable(file)
    